# Volume 1. Lexicon And Readout

Question: after we form several assemblies in one area, can we read a new probe assembly back as the closest known label?

This is the first bridge from neural activity to symbols. The labels here are deliberately humble: `red`, `blue`, and `green` are just names attached to stored assembly snapshots. Readout means "which stored trace does this new trace most resemble?"

In [ ]:
from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import (
    build_lexicon,
    fuzzy_readout,
    project,
    readout_all,
)

Keep the setup visible. We use one area, three word-like labels, and one stimulus per label. Nothing semantic is being learned here; the lesson is the mechanics of storing and comparing assemblies.

In [ ]:
WORDS = ["red", "blue", "green"]
N = 3_000
K = 50

brain = Brain(p=0.05, save_winners=True, seed=11, engine="numpy_sparse")
brain.add_area("LEX", n=N, k=K, beta=0.08)

stimuli = {word: f"stim_{word}" for word in WORDS}
for stimulus in stimuli.values():
    brain.add_stimulus(stimulus, K)

`build_lexicon` projects each stimulus into the same area and stores an assembly snapshot for each label.

In [ ]:
lexicon = build_lexicon(brain, "LEX", WORDS, stimuli, rounds=6)
print({word: len(assembly) for word, assembly in lexicon.items()})

Now form a fresh probe for `red` and rank the stored assemblies by overlap.

In [ ]:
probe = project(brain, stimuli["red"], "LEX", rounds=6)
for word, score in readout_all(probe, lexicon):
    print(f"{word:>5}: {score:.3f}")

print("readout:", fuzzy_readout(probe, lexicon, threshold=0.5))

What to notice: readout is just overlap against stored assemblies. If the best score falls below the threshold, `fuzzy_readout` returns `None` instead of pretending it recognized the probe.

Try next: raise the threshold to `0.95`, then lower it to `0.1`. You are not tuning for glory; you are learning what confidence means when recognition is overlap-based.